In [1]:
from pypdf import PdfReader
reader = PdfReader("test123.pdf")
print(len(reader.pages))
text=""
for page in reader.pages:
    text+=page.extract_text()
print(len(text))
print(text[:500])

82
156905
Savitribai Phule Pune University, Pune
Maharashtra, India
Faculty of Science and Technology
National Education Policy (NEP) - 2020 Compliant Curriculum
SE - Second Year Engineering (2024 Pattern) in
Computer Engineering
(With effect from Academic Year 2025-26)
www.unipune.ac.in/Contents
Abbreviation 1
Preface by Board of Studies 2
Program Educational Objectives 3
Knowledge and Attitude Profile (WK) 4
Program Outcomes 5
General Rules 7
Curriculum Structure - Semester III 10
Curriculum Structure -


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter( chunk_overlap=50, chunk_size=500)
chunks = splitter.split_text(text)
print(len(chunks))
print(chunks[0])
print("abaidbcaebcjabcjacbkabckjabckjabcjbkjcbajkscbjkasbckajbckjasbkjsabcjk")
print(chunks[1])

347
Savitribai Phule Pune University, Pune
Maharashtra, India
Faculty of Science and Technology
National Education Policy (NEP) - 2020 Compliant Curriculum
SE - Second Year Engineering (2024 Pattern) in
Computer Engineering
(With effect from Academic Year 2025-26)
www.unipune.ac.in/Contents
Abbreviation 1
Preface by Board of Studies 2
Program Educational Objectives 3
Knowledge and Attitude Profile (WK) 4
Program Outcomes 5
General Rules 7
Curriculum Structure - Semester III 10
abaidbcaebcjabcjacbkabckjabckjabcjbkjcbajkscbjkasbckajbckjasbkjsabcjk
Curriculum Structure - Semester III 10
Curriculum Structure - Semester IV 11
Semester - III Courses 12
Data Structures 13
Object Oriented programming and Computer Graphics 16
Operating Systems 19
Data Structures Laboratory 22
Object Oriented Programming and Computer Graphics Laboratory 27
Digital Electronics and Logic Design 31
Entrepreneurship Development 34
Universal Human Values and Professional Ethics 40
Community Engagement Project 43
Seme

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings( model_name="sentence-transformers/all-MiniLM-L6-v2")
sample_vector = embeddings.embed_query(chunks[0])
print(len(sample_vector))
print(sample_vector[:10])


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

384
[-0.06607112288475037, 0.039086319506168365, -0.005402547772973776, -0.027216127142310143, -0.040370404720306396, -0.049890946596860886, -0.0785532295703888, 0.05882703885436058, -0.08338236808776855, -0.027858931571245193]


In [4]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_texts(chunks,embeddings)

C:\Users\Raaghav\AppData\Local\Temp\ipykernel_30088\3295633219.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [5]:
results = vectorstore.similarity_search("What is taught in Operating Systems", k=3)
for r in results:
    print(r.page_content)
    print("---")

End-Semester: 70 Marks
,
Prerequisite Courses : Data Structure, Digital Electronics
Companion Course : Computer Organization and Microprocessors
Course Objectives: The course aims to:
1. To understand the fundamental concepts, types, and structures of Operating Systems.
2. To understand and analyze process management concepts.
3. To identify and solve concurrency and deadlock in the operating system.
4. Explore the various techniques of memory management.
---
5. Understand I/O management, disk scheduling, and file systems.
Course Outcomes: Upon successful completion of this course, students will be able to:
• CO1: Analyze the fundamentals of Operating Systems, including types, structures, system calls,
and basic Linux commands.
• CO2: Apply process scheduling and synchronization to optimize CPU utilization in modern
operating systems.
• CO3: Identify the mechanism for dealing with deadlocks and concurrency concerns.
---
• CO4: Apply techniques of memory management to solve memory manag

In [6]:

retriever = vectorstore.as_retriever(search_kwargs={"k":3})


In [7]:
from langchain_core.prompts import PromptTemplate
prompt_template = """You are a helpful assistant answering questions about a college syllabus document.
Use only the following context to answer the question. If the answer isn't in the context, say you don't know — do not make up an answer.

Context:
{context}

Question:
{question}

Answer:"""
prompt = PromptTemplate(template=prompt_template , input_variables=["context","question"])

In [8]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()
llm = ChatGroq(groq_api_key=os.getenv("GROQ_API_KEY"), model = "openai/gpt-oss-20b")

In [9]:
from langchain_classic.chains import RetrievalQA
qa_chain = RetrievalQA.from_chain_type( llm=llm, retriever=retriever, chain_type_kwargs={"prompt":prompt})

In [10]:
response = qa_chain.invoke("How many marks is the end-semester exam worth for Operating Systems?")
print(response["result"])

I’m sorry, but I don’t know.


In [11]:
response2 = qa_chain.invoke("What is the capital of France?")
print(response2["result"])

I don't know.


In [12]:
import hashlib

def get_file_hash(file_bytes):
    return hashlib.sha256(file_bytes).hexdigest()

with open("test123.pdf", "rb") as f:
    file_bytes = f.read()

file_hash = get_file_hash(file_bytes)
print(file_hash)

ff6ca06b0bdd62b0c0ed46fc0e74cca312594d504650c9aa30795207ea4118a3


In [13]:
from supabase import create_client
import os
from dotenv import load_dotenv

load_dotenv()

supabase = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))

In [14]:

existing = supabase.table("documents").select("*").eq("file_hash", file_hash).execute()
print(existing.data)

[]


In [15]:
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import SupabaseVectorStore

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = SupabaseVectorStore(
    client=supabase,
    embedding=embeddings,
    table_name="document_chunks",
    query_name="match_document_chunks"
)

if len(existing.data) > 0:
    print("Already processed — skipping re-indexing")
else:
    print("New file — processing and uploading")

    reader = PdfReader("test123.pdf")
    text = ""
    for page in reader.pages:
        text += page.extract_text()

    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_text(text)

    metadatas = [{"file_hash": file_hash} for _ in chunks]
    vectorstore.add_texts(chunks, metadatas=metadatas)

    supabase.table("documents").insert({
        "filename": "test123.pdf",
        "file_hash": file_hash,
        "chunk_count": len(chunks)
    }).execute()

    print(f"Uploaded {len(chunks)} chunks")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

New file — processing and uploading
Uploaded 347 chunks


In [16]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3, "filter": {"file_hash": file_hash}}
)

In [17]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt}
)

response = qa_chain.invoke("How many marks is the end-semester exam worth for Operating Systems?")
print(response["result"])

The end‑semester exam for Operating Systems is worth **70 marks**.
